
#  OpenMP Programming Tutorial and Hands-on


Enter your name and student ID.

 * Name: HU Yuefeng
 * Student ID: 48226639



# 1. OpenMP
* <a href="http://openmp.org/" target="_blank" rel="noopener">OpenMP</a> is the de fact programming model for multicore environment
* More recently, it supports GPU offloading
* We are going to learn OpenMP both for CPU (multicore) and GPU programming
* See <a href="https://www.openmp.org/spec-html/5.0/openmp.html" target="_blank" rel="noopener">the spec</a>


# 2. Compilers
* We use [NVIDIA HPC SDK ver. 22.7](https://docs.nvidia.com/hpc-sdk/index.html) (`nvc` and `nvc++`) and [LLVM ver. 15.0](https://llvm.org/) (`clang` and `clang++`) for C/C++ compilers, as they support OpenMP GPU offloading

## 2-1. Set up NVIDIA HPC SDK
Execute this before you use NVIDIA HPC SDK

In [662]:
export PATH=/opt/nvidia/hpc_sdk/Linux_x86_64/22.7/compilers/bin:$PATH

Check if it works (check if full paths of nvc/nvc++ are shown)

In [663]:
which nvc
which nvc++

/opt/nvidia/hpc_sdk/Linux_x86_64/22.7/compilers/bin/nvc
/opt/nvidia/hpc_sdk/Linux_x86_64/22.7/compilers/bin/nvc++


## 2-2. Set up LLVM
Execute this before you use LLVM

In [664]:
export PATH=/home/share/llvm/bin:$PATH

Check if it works (check if full paths of nvc/nvc++ are shown)

In [665]:
which clang
which clang++

/home/share/llvm/bin/clang
/home/share/llvm/bin/clang++


# 3. Compiling and running OpenMP programs
* Summary
  * `clang`/`clang++` : give `-fopenmp` option
  * `nvc`/`nvc++` : give `-mp` option
  * Set `OMP_NUM_THREADS` environment variable when running the executable

In [727]:
%%writefile omp_hello.c
#include <omp.h>
#include <stdio.h>

int main() {
  printf("hello\n");
#pragma omp parallel
  printf("world\n");
  printf("good bye\n");
  return 0;
}

Overwriting omp_hello.c


* Compiling with clang
* Add `-fopenmp` option to compile OpenMP programs
* Other generally useful options
  * `-Wall` warns many suspicous code
  * `-O3` maximally optimize code for performance

In [667]:
clang -fopenmp omp_hello.c -o omp_hello_clang

* Compiling with nvc
* Add `-mp` option to compile OpenMP programs
* Other generally useful options
  * `-Wall` warns many suspicous code
  * `-O4` maximally optimizes code for performance

In [668]:
nvc -mp omp_hello.c -o omp_hello_nvc

* Running

* Set environment variable `OMP_NUM_THREADS` to the number of threads created by `#pragma omp parallel`

In [669]:
OMP_NUM_THREADS=3 ./omp_hello_clang

hello
world
world
world
good bye


In [670]:
OMP_NUM_THREADS=3 ./omp_hello_nvc

hello
world
world
world
good bye


# <font color="green"> Problem 1 :  Change the number of threads</font>
* Execute them with various numbers of threads and see what happens

In [671]:
OMP_NUM_THREADS=30 ./omp_hello_clang

hello
world
world
world
world
world
world
world
world
world
world
world
world
world
world
world
world
world
world
world
world
world
world
world
world
world
world
world
world
world
world
good bye


In [672]:
OMP_NUM_THREADS=10 ./omp_hello_nvc

hello
world
world
world
world
world
world
world
world
world
world
good bye


# 4. `#pragma omp parallel`
* [#pragma omp parallel](https://www.openmp.org/spec-html/5.0/openmpse14.html#x54-800002.6) creates a _team_ of threads, each of which executes the next statement
* Note that only the statement that is right next to the pragma is executed by the team of threads
* Of course, the statement can be a compound statement and/or include a function call, so each thread can actually execute arbitrary number of statements
* See [Determining the Number of Threads for a parallel Region](https://www.openmp.org/spec-html/5.0/openmpsu35.html#x55-880002.6.1) for more details on the number of threads created by `#pragma omp parallel`

# <font color="green"> Problem 2 :  Executing multiple statements by threads</font>
* Change following program so that both "world" and "good bye" are printed as many times as the number of threads

In [728]:
%%writefile omp_hello.c
#include <omp.h>
#include <stdio.h>

int main() {
  printf("hello\n");
#pragma omp parallel
{
  printf("world\n");
  printf("good bye\n");
}
  return 0;
}

Overwriting omp_hello.c


* Below, choose `clang` or `nvc` depending on your taste by commenting out the other one
* Below, I chose `clang` by commenting out `nvc`

In [729]:
clang -fopenmp omp_hello.c -o omp_hello
# nvc -mp omp_hello.c -o omp_hello

In [730]:
OMP_NUM_THREADS=3 ./omp_hello

hello
world
good bye
world
good bye
world
good bye


# 5. `omp_get_num_threads()` and `omp_get_thread_num()`
* When threads are executing a statement with `#pragma omp parallel`,
  * they are said to be in a _parallel region_
  * they are called a _team_ of threads

* While a thread is executing a parallel region,
  * [omp_get_num_threads()](https://www.openmp.org/spec-html/5.0/openmpsu111.html#x148-6450003.2.2) returns the number of threads in the team 
  * [omp_get_thread_num()](https://www.openmp.org/spec-html/5.0/openmpsu113.html#x150-6570003.2.4) returns the unique id of the calling thread within the team (0, 1, ..., the number threads in the team - 1)
* You need `#include <omp.h>` to use these functions or any OpenMP API functions, for that matter


# <font color="green"> Problem 3 :  Using `omp_get_num_threads()` and `omp_get_thread_num()`</font>
* Change following program so that each thread prints its id in the team and the number of threads in the team, like this.  The exact order of lines may differ.  Strictly speaking, even characters in two lines can be mixed into a single line.

```
hello
0/5 world
4/5 world
1/5 world
3/5 world
2/5 world
good bye
```

In [676]:
%%writefile omp_hello_id.c
#include <stdio.h>
#include <omp.h>

int main() {
  printf("hello\n");
#pragma omp parallel
{
  int t = omp_get_thread_num(); // id
  int nt = omp_get_num_threads(); // number of threads
  printf("%d/%d world\n", t, nt);
}
  
  printf("good bye\n");
  return 0;
}

Overwriting omp_hello_id.c


In [677]:
clang -fopenmp omp_hello_id.c -o omp_hello_id
# nvc -mp omp_hello_id.c -o omp_hello_id

In [678]:
OMP_NUM_THREADS=5 ./omp_hello_id

hello
0/5 world
2/5 world
3/5 world
1/5 world
4/5 world
good bye


# 6. `#pragma omp for`
* `#pragma omp parallel` merely creates a team of threads executing the same statement
* In this sense, `#pragma omp parallel` alone cannot make a program run faster with multiple cores
* A program can be made faster only when you _divide_ the work among threads (work-sharing)
* `#pragma omp for` lets you divide iterations of a loop into threads created by `#pragma omp parallel`

# <font color="green"> Problem 4 :  How does `#pragma omp for` divide iterations to threads?</font>
* Execute the following cell and observe which iteration is executed by which thread
* Based on the observation, change the number of iterations and threads and predict the mapping between iterations and threads

In [679]:
%%writefile omp_for.c
#include <stdio.h>
#include <unistd.h>
#include <omp.h>

int main() {
#pragma omp parallel
  {
    printf("I am thread %d in a team of %d threads\n",
           omp_get_thread_num(), omp_get_num_threads());
#pragma omp for
    for (int i = 0; i < 24; i++) {
      usleep(100 * 1000 * i);
      printf("iteration %d executed by thread %d\n", i, omp_get_thread_num());
      fflush(stdout);
    }
  }
  return 0;
}


Overwriting omp_for.c


In [680]:
clang -fopenmp omp_for.c -o omp_for
# nvc -mp omp_for.c -o omp_for

In [681]:
OMP_NUM_THREADS=6 ./omp_for

I am thread 0 in a team of 6 threads
I am thread 3 in a team of 6 threads
I am thread 4 in a team of 6 threads
I am thread 5 in a team of 6 threads
iteration 0 executed by thread 0
I am thread 1 in a team of 6 threads
I am thread 2 in a team of 6 threads
iteration 1 executed by thread 0
iteration 2 executed by thread 0
iteration 4 executed by thread 1
iteration 3 executed by thread 0
iteration 8 executed by thread 2
iteration 5 executed by thread 1
iteration 12 executed by thread 3
iteration 6 executed by thread 1
iteration 16 executed by thread 4
iteration 9 executed by thread 2
iteration 20 executed by thread 5
iteration 7 executed by thread 1
iteration 13 executed by thread 3
iteration 10 executed by thread 2
iteration 17 executed by thread 4
iteration 11 executed by thread 2
iteration 14 executed by thread 3
iteration 21 executed by thread 5
iteration 18 executed by thread 4
iteration 15 executed by thread 3
iteration 22 executed by thread 5
iteration 19 executed by thread 4
iterat

## 6-1. for loops allowed by `#pragma omp for`
* There is a severe syntax restriction on the kind of for loops `#pragma omp for` can apply for
* See [Canonical Loop Form](https://www.openmp.org/spec-html/5.0/openmpsu40.html#x63-1260002.9.1) for the spec
* In short, it should look like `for (var = _init_; var < _limit_; var += _inc_)` where _init_, _limit_, and _inc_ are all loop-invariant (do not change throughout the loop)

## 6-2. Combined pragma (parallel + for)
* `#pragma omp parallel` and `#pragma omp for` are often used together
* If `#pragma omp for` immediately follows `#pragma omp parallel`, they can be combined into a single pragma `#pragma omp parallel for`

In [682]:
%%writefile omp_parallel_for.c
#include <stdio.h>
#include <unistd.h>
#include <omp.h>

int main() {
  double t0 = omp_get_wtime();
#pragma omp parallel for
  for (int i = 0; i < 24; i++) {
    usleep(100 * 1000 * i);     /* sleep 100 x i milliseconds */
    printf("iteration %d executed by thread %d\n", i, omp_get_thread_num());
    fflush(stdout);
  }
  double t1 = omp_get_wtime();
  printf("%f sec\n", t1 - t0);
  return 0;
}


Overwriting omp_parallel_for.c


In [683]:
clang -fopenmp omp_parallel_for.c -o omp_parallel_for
# nvc -mp omp_parallel_for.c -o omp_parallel_for

In [684]:
OMP_NUM_THREADS=4 ./omp_parallel_for

iteration 0 executed by thread 0
iteration 1 executed by thread 0
iteration 2 executed by thread 0
iteration 6 executed by thread 1
iteration 3 executed by thread 0
iteration 4 executed by thread 0
iteration 12 executed by thread 2
iteration 7 executed by thread 1
iteration 5 executed by thread 0
iteration 18 executed by thread 3
iteration 8 executed by thread 1
iteration 13 executed by thread 2
iteration 9 executed by thread 1
iteration 19 executed by thread 3
iteration 14 executed by thread 2
iteration 10 executed by thread 1
iteration 11 executed by thread 1
iteration 15 executed by thread 2
iteration 20 executed by thread 3
iteration 16 executed by thread 2
iteration 21 executed by thread 3
iteration 17 executed by thread 2
iteration 22 executed by thread 3
iteration 23 executed by thread 3
12.316868 sec


# 7. Scheduling a work-sharing for loop
* As you witnessed, the default scheduling policy in our environment (may be implementation dependent) seems static scheduling (assign roughly the same number of contiguous iterations to each thread)
* Is it enough? Clearly, it does not do a good job when iterations take a different amount of time
* You can change the policy by [schedule clause](https://www.openmp.org/spec-html/5.0/openmpsu41.html#x64-1290002.9.2)

# <font color="green"> Problem 5 :  Specifying the scheduling policy by schedule clause</font>
* In the following (artificial) loop, iteration _i_ roughly sleeps for (100 x _i_) milliseconds and this is almost exactly the time it takes
1. reason about the executing time of the parallel for loop with the default (static) scheduling policy
1. vary the scheduling policy and reason about their execution times

In [685]:
%%writefile omp_schedule.c
#include <stdio.h>
#include <unistd.h>
#include <omp.h>

int main() {
  printf("-------------- STATIC START--------------\n");
  double t0 = omp_get_wtime();
  /* ----- add schedule clause below ----- */
#pragma omp parallel for schedule(static)
  for (int i = 0; i < 24; i++) {
    usleep(100 * 1000 * i);     /* sleep 100 x i milliseconds */
    printf("iteration %d executed by thread %d\n", i, omp_get_thread_num());
    fflush(stdout);
  }
  double t1 = omp_get_wtime();
  printf("%f sec\n", t1 - t0);
  printf("-------------- STATIC END--------------\n\n");

  printf("-------------- DYNAMIC START--------------\n");
  double t2 = omp_get_wtime();
  /* ----- add schedule clause below ----- */
#pragma omp parallel for schedule(dynamic)
  for (int i = 0; i < 24; i++) {
    printf("iteration %d start executing by thread %d\n", i, omp_get_thread_num());
    usleep(100 * 1000 * i);     /* sleep 100 x i milliseconds */
    printf("iteration %d executed by thread %d\n", i, omp_get_thread_num());
    fflush(stdout);
  }
  double t3 = omp_get_wtime();
  printf("%f sec\n", t3 - t2);
  printf("-------------- DYNAMIC END--------------\n");

  return 0;
}


Overwriting omp_schedule.c


In [686]:
clang -fopenmp omp_schedule.c -o omp_schedule
# nvc -mp omp_schedule.c -o omp_schedule

In [687]:
OMP_NUM_THREADS=4 ./omp_schedule

-------------- STATIC START--------------
iteration 0 executed by thread 0
iteration 1 executed by thread 0
iteration 2 executed by thread 0
iteration 6 executed by thread 1
iteration 3 executed by thread 0
iteration 4 executed by thread 0
iteration 12 executed by thread 2
iteration 7 executed by thread 1
iteration 5 executed by thread 0
iteration 18 executed by thread 3
iteration 8 executed by thread 1
iteration 13 executed by thread 2
iteration 9 executed by thread 1
iteration 19 executed by thread 3
iteration 14 executed by thread 2
iteration 10 executed by thread 1
iteration 11 executed by thread 1
iteration 15 executed by thread 2
iteration 20 executed by thread 3
iteration 16 executed by thread 2
iteration 21 executed by thread 3
iteration 17 executed by thread 2
iteration 22 executed by thread 3
iteration 23 executed by thread 3
12.318841 sec
-------------- STATIC END--------------

-------------- DYNAMIC START--------------
iteration 18 start executing by thread 3
iteration 0 s

* What should be the execution time with the default (static) policy?
* What should be the execution time with the dynamic policy?
* Compare them with what you observed 

## My Answer:

#### Static Policy
Each iteration will take 0.1 * i seconds. For static policy, thread 0 will be assigned the iteration 0 to 5, thread 1 will be assigned the iteration 6 to 11, thread 2 will be assigned the iteration 12 to 17, and thread 3 will get iteration 18 to 23. So apparently the longest thread will be thread 3, so the total execution time will be 1.8+1.9+2.0+2.1+2.2+2.3=12.3 seconds.

#### Dynamic Policy
At the beginning, these 24 iterations will be assigned like this (same as static policy):
- Thread 0: iteration 0, 1, 2, 3, 4, 5
- Thread 1: iteration 6, 7, 8, 9, 10, 11
- Thread 2: iteration 12, 13, 14, 15, 16, 17
- Thread 3: iteration 18, 19, 20, 21, 22, 23
    
But each thread will repeat fetching chunk iterations when all its task is finished, and this is when ***[work stealing](https://en.wikipedia.org/wiki/Work_stealing)*** happens. The thread will "steal" the task of other thread's task stack ***from the bottom***. That's why thread 0 will steal iteration 11 from thread 1 first, and then steal iteration 10 from thread 1, and thread 1 steal iteration 17 and 23 from other thread, etc.
    
The real execution order of dynamic schedule is:
- Thread 0: iteration 0, 1, 2, 3, 4, 5, 11, 10, 16, 22 = 7.4 sec
- Thread 1: iteration 6, 7, 8, 9, 17, 23 = 7.0 sec
- Thread 2: iteration 12, 13, 14, 15, 21 = ***7.5 sec***
- Thread 3: iteration 18, 19, 20 = 57 sec

The real execution time comes from thread 2.
    

# 8. Collapse clause
* `#pragma omp for` can specify a [collapse clause](https://www.openmp.org/spec-html/5.0/openmpsu41.html#x64-1290002.9.2) to apply work-sharing for a limited type of nested loops
* With a `#pragma omp for clause(2)`, OpenMP considers the doubly-nested loop that comes after this pragma the subject of work-sharing (i.e., distribute iterations of the doubly-nested loop to threads); you must have a _perfectly-nested_, rectangular doubly-nested loop after this clause
* A perfectly-nested loop is a nested loop whose outer loops (all loops except for the innermost one) do not have any statement except the inner loop. e.g.
```
for (i = 0; i < 100; i++) {
  for (j = 0; j < 100; j++) {
    S(i,j);
  }
}
```
is perfectly nested whereas
```
for (i = 0; i < 100; i++) {
  S;
  for (j = 0; j < 100; j++) {
    T;
  }
}
```
is not.  
* A perfectly nested loop is conceptually a flat loop with a mechanical transformation.
```
for (ij = 0; ij < 100 * 100; ij++) {
  i = ij / 100;
  j = ij % 100;
  S(i,j);
}
```
* A rectangular loop is a loop whose iteration counts of inner loops never depend on outer loops.  For example,
```
for (i = 0; i < 100; i++) {
  for (j = 0; j < i; j++) {
    S(i,j);
  }
}
```
is not a rectangular loop.
* Generally speaking, OpenMP `#pragma omp parallel` + `#pragma omp for` cannot handle nested parallelism very well, but collapse clause alleviates the problem to some extent
* Consider using tasks below for more general form of nested parallelism

# <font color="green"> Problem 6 :  Apply collapse and schedule</font>
* Apply collapse and schedule to the following loop
* Trick: write `schedule(runtime)` and you can change the scheduling policy at execution time by setting environment variable `OMP_SCHEDULE=` in the command line. See [OMP_SCHEDULE environment variable](https://www.openmp.org/spec-html/5.0/openmpsu41.html#x64-1370002.9.2.1) for details
* Reason about the execution time of various schedule policies and with/without collapse

In [688]:
%%writefile omp_collapse.c
//% file: omp_collapse_1.c
//% cmd: gcc -Wall -Wextra -fopenmp -fopenmp-simd -O3 -I. -o omp_collapse_1 omp_collapse_1.c -lm

#include <stdio.h>
#include <unistd.h>
#include <omp.h>

int main() {
  double t0 = omp_get_wtime();
  /* apply collapse and schedule */
#pragma omp parallel for schedule(runtime) collapse(2)
  for (int i = 0; i < 5; i++) {
    for (int j = 0; j < 5; j++) {
      usleep(100 * 1000 * (i + j));
      printf("iteration (%d, %d) executed by thread %d\n", i, j, omp_get_thread_num());
      fflush(stdout);
    }
  }
  double t1 = omp_get_wtime();
  printf("%f sec\n", t1 - t0);
  return 0;
}



Overwriting omp_collapse.c


In [689]:
clang -fopenmp omp_collapse.c -o omp_collapse
# nvc -mp omp_collapse.c -o omp_collapse

In [690]:
OMP_SCHEDULE=dynamic OMP_NUM_THREADS=5 ./omp_collapse

iteration (0, 0) executed by thread 0
iteration (0, 1) executed by thread 2
iteration (1, 0) executed by thread 4
iteration (0, 2) executed by thread 1
iteration (0, 3) executed by thread 3
iteration (1, 1) executed by thread 2
iteration (0, 4) executed by thread 0
iteration (1, 2) executed by thread 4
iteration (2, 0) executed by thread 2
iteration (1, 3) executed by thread 1
iteration (2, 1) executed by thread 0
iteration (1, 4) executed by thread 3
iteration (2, 2) executed by thread 4
iteration (2, 3) executed by thread 2
iteration (3, 0) executed by thread 0
iteration (2, 4) executed by thread 1
iteration (3, 1) executed by thread 3
iteration (3, 2) executed by thread 4
iteration (4, 0) executed by thread 1
iteration (3, 3) executed by thread 2
iteration (4, 1) executed by thread 3
iteration (3, 4) executed by thread 0
iteration (4, 2) executed by thread 4
iteration (4, 3) executed by thread 1
iteration (4, 4) executed by thread 2
2.415151 sec


# 9. Tasks
* Task is a more general mechanism to extract parallelism and distribute computation (called a task) _dynamically_ to threads in a team created by `#pragma omp parallel`
* A thread can create a task at any point in the execution of a parallel region and they are dispatched to available threads at runtime
* As a thread can create a task at any point, a task can create another task. that is, parallelism can be arbitrarily nested and the number of tasks can be difficult to predict (unlike the number of iterations of a for loop)
* A common pattern
  1. enter a parallel region by `#pragma omp parallel`
  1. ensure the statement is executed by only a single (root) thread with [#pragma omp master](https://www.openmp.org/spec-html/5.0/openmpse24.html#x118-4380002.16)
  1. create tasks at any point by [#pragma omp task](https://www.openmp.org/spec-html/5.0/openmpsu46.html#x70-2000002.10.1)
  1. a task waits for tasks it created to finish by [#pragma omp taskwait](https://www.openmp.org/spec-html/5.0/openmpsu93.html#x124-4690002.17.5)

* Let's see the effect of `#pragma omp master` first (without creating any task)

In [691]:
%%writefile omp_master.c
#include <stdio.h>
#include <unistd.h>
#include <omp.h>

int main() {
  double t0 = omp_get_wtime();
#pragma omp parallel
  {
#pragma omp master
    printf("inside the master pragma: I am thread %d of a team of %d threads\n",
           omp_get_thread_num(), omp_get_num_threads());
    printf("out of the master pragma: I am thread %d of a team of %d threads\n",
           omp_get_thread_num(), omp_get_num_threads());
  }
  double t1 = omp_get_wtime();
  printf("%f sec\n", t1 - t0);
  return 0;
}


Overwriting omp_master.c


In [692]:
clang -fopenmp omp_master.c -o omp_master
# nvc -mp omp_master.c -o omp_master

In [693]:
OMP_NUM_THREADS=3 ./omp_master

inside the master pragma: I am thread 0 of a team of 3 threads
out of the master pragma: I am thread 0 of a team of 3 threads
out of the master pragma: I am thread 2 of a team of 3 threads
out of the master pragma: I am thread 1 of a team of 3 threads
0.020775 sec


* Since this is a common idiom, they can be combined into one pragma
* The program below creates a parallel region whose entire region is executed only by the master and thus does not serve any useful purpose but mere a demonstration of the feature

In [694]:
%%writefile omp_parallel_master.c
#include <stdio.h>
#include <unistd.h>
#include <omp.h>

int main() {
  double t0 = omp_get_wtime();
#pragma omp parallel master
  printf("I am thread %d of a team of %d threads\n",
         omp_get_thread_num(), omp_get_num_threads());
  double t1 = omp_get_wtime();
  printf("%f sec\n", t1 - t0);
  return 0;
}


Overwriting omp_parallel_master.c


In [695]:
clang -fopenmp omp_parallel_master.c -o omp_parallel_master
# nvc -mp omp_parallel_master.c -o omp_parallel_master

In [696]:
OMP_NUM_THREADS=3 ./omp_parallel_master

I am thread 0 of a team of 3 threads
0.015985 sec


* Let's create a few tasks now

In [697]:
%%writefile omp_task.c
#include <stdio.h>
#include <unistd.h>
#include <omp.h>

int main() {
  double t0 = omp_get_wtime();
#pragma omp parallel master
  {
    printf("I am thread %d of a team of %d threads\n",
           omp_get_thread_num(), omp_get_num_threads());
#pragma omp task
    {
      printf("task A executed by %d of %d\n", omp_get_thread_num(), omp_get_num_threads());
      usleep(500 * 1000);
    }
#pragma omp task
    {
      printf("task B executed by %d of %d\n", omp_get_thread_num(), omp_get_num_threads());
      usleep(1000 * 1000);
    }
#pragma omp taskwait
    printf("two tasks done, executed by %d of %d\n", omp_get_thread_num(), omp_get_num_threads());
  }
  double t1 = omp_get_wtime();
  printf("%f sec\n", t1 - t0);
  return 0;
}


Overwriting omp_task.c


In [698]:
clang -fopenmp omp_task.c -o omp_task
# nvc -mp omp_task.c -o omp_task

In [699]:
OMP_NUM_THREADS=3 ./omp_task

I am thread 0 of a team of 3 threads
task A executed by 1 of 3
task B executed by 0 of 3
two tasks done, executed by 0 of 3
1.014234 sec


* Tasks are particularly good at parallel recursions, as the following program demonstrates
* This is a common pattern that appears in many algorithms, particularly divide-and-conquer algorithms

In [700]:
%%writefile omp_rec_task.c
#include <stdio.h>
#include <unistd.h>
#include <omp.h>

void recursive_tasks(int n, int tid) {
  printf("task %d by %d of %d\n",
         tid, omp_get_thread_num(), omp_get_num_threads());
  fflush(stdout);
  if (n == 0) {
    usleep(300 * 1000);
  } else {
#pragma omp task
    recursive_tasks(n - 1, 2 * tid + 1);
#pragma omp task
    recursive_tasks(n - 1, 2 * tid + 2);
#pragma omp taskwait
  }
}
int main() {
  double t0 = omp_get_wtime();
#pragma omp parallel master
  {
    recursive_tasks(5, 0);
  }
  double t1 = omp_get_wtime();
  printf("%f sec\n", t1 - t0);
  return 0;
}


Overwriting omp_rec_task.c


In [701]:
clang -fopenmp omp_rec_task.c -o omp_rec_task
# nvc -mp omp_rec_task.c -o omp_rec_task

In [702]:
OMP_NUM_THREADS=10 ./omp_rec_task

task 0 by 0 of 10
task 2 by 0 of 10
task 1 by 2 of 10
task 6 by 0 of 10
task 5 by 8 of 10
task 13 by 6 of 10
task 11 by 9 of 10
task 27 by 5 of 10
task 4 by 3 of 10
task 12 by 1 of 10
task 28 by 6 of 10
task 58 by 6 of 10
task 14 by 0 of 10
task 30 by 0 of 10
task 62 by 0 of 10
task 3 by 4 of 10
task 56 by 5 of 10
task 55 by 7 of 10
task 9 by 2 of 10
task 20 by 2 of 10
task 42 by 2 of 10
task 25 by 8 of 10
task 52 by 8 of 10
task 24 by 9 of 10
task 50 by 9 of 10
task 8 by 4 of 10
task 18 by 4 of 10
task 38 by 4 of 10
task 26 by 1 of 10
task 54 by 1 of 10
task 10 by 3 of 10
task 22 by 3 of 10
task 46 by 3 of 10
task 57 by 6 of 10
task 61 by 0 of 10
task 53 by 5 of 10
task 29 by 7 of 10
task 41 by 2 of 10
task 60 by 7 of 10
task 51 by 8 of 10
task 37 by 4 of 10
task 49 by 9 of 10
task 45 by 3 of 10
task 19 by 6 of 10
task 40 by 6 of 10
task 59 by 0 of 10
task 39 by 5 of 10
task 21 by 1 of 10
task 23 by 8 of 10
task 48 by 8 of 10
task 44 by 1 of 10
task 17 by 4 of 10
task 36 by 4 of 10
ta

# <font color="green"> Problem 7 :  A quiz about recursive tasks</font>
* Answer the following questions
* How many tasks are created by `recursive_tasks(n, 0)`?  Include the caller of `recursive_tasks(n, 0)` as a task.  i.e., consider `recursive_tasks(0, 0)` creates one task
* How many of them are leaf tasks?

#### My Answer

1. In fact, the task recursive list looks exactly like a binary tree, so there is `2 ^ (n + 1) - 1` threads. For example, for the code below, recursive_tasks(5, 0) created **2 ^ (5 + 1) - 1 = 63** tasks.
2. just like the binary tree, there will be `2 ^ n` leaf tasks.

* Approximately what is the ideal execution time of `recursive_tasks(5, 0)` when using 10 threads?
* Compare it with what you observed

#### My Answer

There are 32 leaf tasks and 10 threads for recursive_tasks(5, 0), 32 = 10 * 3 + 2, so after three 10-thread parallel circle, there still remains 2 tasks, and each leaf task takes 0.3 sec, so the ideal execution time of recursive_tasks(5, 0) is 1.2 seconds. And the real result is about 1.22 seconds, which is close to the ideal time.

* Name an algorithm or two for which recursive tasks will be useful for parallelizing it and explain why you think so

#### My Answer

Binary search tree or balanced search tree like red-black balanced search tree, but only for searching tasks. In the insertion or deletion function of BST (binary search tree or balanced search tree), the structure of the tree will be changed during the processing, and the order of the recursive tasks must be a certain order, parallelizing the insertion or deletion functions will mess up the order and make the result inaccurate. But for searching tasks, the structure of the tree won't be changed during the processing, and the order of the recursive tasks doesn't have to be a certain order, so I think parallel programming could be useful for these kinds of searching tasks.

# 10. Taskloop
* As you can easily imagine, tasks can handle general nested loops if they can handle recursions
* Recent OpenMP actually has a construct just for that, which is [#pragma omp taskloop](https://www.openmp.org/spec-html/5.0/openmpsu47.html#x71-2080002.10.2)
* Here is a demonstration showing it can handle non perfectly-nested loops

In [703]:
%%writefile omp_taskloop.c
//% file: omp_taskloop_1.c
//% cmd: gcc -Wall -Wextra -fopenmp -fopenmp-simd -O3 -I. -o omp_taskloop_1 omp_taskloop_1.c -lm
//% cmd: OMP_NUM_THREADS=5 ./omp_taskloop_1

#include <stdio.h>
#include <unistd.h>
#include <omp.h>

int main() {
  double t0 = omp_get_wtime();
#pragma omp parallel master
#pragma omp taskloop
  for (int i = 0; i < 5; i++) {
    printf("i = %d starts\n", i);
    fflush(stdout);
#pragma omp taskloop
    for (int j = 0; j < 5; j++) {
      usleep(100 * 1000 * (i + j));
      printf("iteration (%d, %d) executed by thread %d\n", i, j, omp_get_thread_num());
      fflush(stdout);
    }
  }
  double t1 = omp_get_wtime();
  printf("%f sec\n", t1 - t0);
  return 0;
}


Overwriting omp_taskloop.c


In [704]:
clang -fopenmp omp_taskloop.c -o omp_taskloop
# nvc -mp omp_taskloop.c -o omp_taskloop

In [705]:
OMP_NUM_THREADS=3 ./omp_taskloop

i = 4 starts
i = 0 starts
i = 1 starts
iteration (0, 4) executed by thread 1
iteration (1, 4) executed by thread 2
iteration (0, 3) executed by thread 1
iteration (4, 4) executed by thread 0
iteration (1, 3) executed by thread 2
iteration (0, 2) executed by thread 1
iteration (0, 1) executed by thread 1
iteration (0, 0) executed by thread 1
i = 2 starts
iteration (1, 2) executed by thread 2
iteration (1, 1) executed by thread 2
iteration (4, 3) executed by thread 0
iteration (1, 0) executed by thread 2
i = 3 starts
iteration (2, 4) executed by thread 1
iteration (4, 2) executed by thread 0
iteration (2, 3) executed by thread 1
iteration (3, 4) executed by thread 2
iteration (2, 2) executed by thread 1
iteration (4, 1) executed by thread 0
iteration (3, 3) executed by thread 2
iteration (2, 1) executed by thread 1
iteration (4, 0) executed by thread 0
iteration (2, 0) executed by thread 1
iteration (3, 0) executed by thread 0
iteration (3, 2) executed by thread 2
iteration (3, 1) execut

<font color="red">NOTE:</font>

* Implementing task requires a more general mechanism than work-sharing for statement; the former should be able to distribute tasks generated in the course of execution, whereas the former merely needs to distribute iterations that are easily identifiable at the point of entering `#pragma omp for`, thanks to the ["canonical form" restriction](https://www.openmp.org/spec-html/5.0/openmpsu40.html#x63-1260002.9.1)
* Task scheduling is always dynamic whereas work-sharing for (particularly with static scheduling) gives you more control and predictability about which thread executes which iteration
* This is a reason why two mechanisms which are seemingly redundant exist, besides a historical reason that initially there was not a tasking construct in OpenMP

# 11. Data sharing
* OpenMP is a shared memory programming model, which means threads see updates among each other
* That is, when a thread updates a variable _x_ that is then read by another, the reader thread will see the updated value
* This is the default behavior of OpenMP ([Data Environment](https://www.openmp.org/spec-html/5.0/openmpse27.html#x135-5430002.19) in OpenMP spec)
* It is not always convenient, however
* `#pragma omp parallel` can thus specify whether local variables in the scope (i.e., defined outside the statement) are privatized (i.e., made private to each thread)

# <font color="green"> Problem 8 :  Observe the effect of privatization</font>
* Execute the following and make sense of the output
* Add the `private(x)` clause and observe the difference

In [706]:
%%writefile omp_private.c
#include <stdio.h>
#include <unistd.h>
#include <omp.h>

int main() {
  int x = 123;
  printf("before : x = %d\n", x);
  /* add private(x) clause below and see the difference */
#pragma omp parallel private(x)
  {
    int id = omp_get_thread_num();
    printf("thread %d : x = %d\n", id, x);
  }
  printf("after : x = %d\n", x);
  return 0;
}


Overwriting omp_private.c


In [707]:
clang -fopenmp omp_private.c -o omp_private
# nvc -mp omp_private.c -o omp_private

In [708]:
OMP_NUM_THREADS=10 ./omp_private

before : x = 123
thread 8 : x = 0
thread 2 : x = 0
thread 0 : x = 0
thread 1 : x = -2
thread 4 : x = 0
thread 3 : x = -2
thread 5 : x = -2
thread 7 : x = -2
thread 6 : x = 0
thread 9 : x = -2
after : x = 123


* `private(x)` essentially ignores the original variable `x` defined outside the parallel region and behaves as if a variable of the same name is defined by each thread
* `firstprivate(x)` is like `private(x)`, except that `x` of each thread is initialized by the value of `x` just before entering the parallel region

# <font color="green"> Problem 9 :  Observe the effect of `private` and `firstprivate`</font>
* Execute the following and observe the output
* Add the `private(x)` clause and execute it
* Add the `firstprivate(x)` clause and execute it
* Make sense of the differences

In [709]:
writefile omp_firstprivate.c
#include <stdio.h>
#include <unistd.h>
#include <omp.h>

int main() {
  int x = 123;
  printf("before : x = %d\n", x);
  /* add private(x)/firstprivate(x) clause and see the difference */
#pragma omp parallel firstprivate(x)
  {
    int id = omp_get_thread_num();
    x++;
    printf("thread %d : x = %d\n", id, x);
  }
  printf("after : x = %d\n", x);
  return 0;
}


Overwriting omp_firstprivate.c


In [710]:
clang -fopenmp omp_firstprivate.c -o omp_firstprivate
# nvc -mp omp_firstprivate.c -o omp_firstprivate

In [711]:
OMP_NUM_THREADS=3 ./omp_firstprivate

before : x = 123
thread 0 : x = 124
thread 1 : x = 124
thread 2 : x = 124
after : x = 123


# 12. Race condition
* Execute the above code without private or firstprivate many times
* Observe that the value of `x` after the parallel region is not always 123 + 5 (the number of threads executing the region), even different across runs
* For example, with two threads, the following execution order may cause such a behavior
  1. thread A reads 123
  1. thread B reads 123
  1. thread A writes 124
  1. thread B reads 124
* A similar case occurs whenever a thread's read-followed-by-write is intervened by another thread's update
* More generally, the following situation is called a "race condition" and if there is a race condition in your program, it almost always means your program is broken
  * Two or more threads concurrently access the same variable, and
  * at least one of them writes to it
Here, "concurrently access" means these accesses are not guaranteed to be separated in time by a synchronization primitive

* In all but trivial parallel programs, threads need to communicate with each other to accomplish a task
* Threads _communicate_ by having one thread write to a variable and having another read it
* If we simply do it without any mechanism to guarantee that they are separated in time, it is a race

* Below, we describe three ways to _safely_ communicate among threads without making race conditions

  * `#pragma omp critical`
  * `#pragma omp atomic`
  * reduction

# 13. `#pragma omp critical`
* [#pragma omp critical](https://www.openmp.org/spec-html/5.0/openmpsu89.html#x120-4470002.17.1) guarantees the statement following the pragma is executed not overlapping in time

# <font color="green"> Problem 10 :  Apply `#pragma omp critical`</font>
* Execute the following program a few times and observe the result
* Add `#pragma omp critical` to the statement `x++` and observe the result

In [712]:
%%writefile omp_critical.c
#include <stdio.h>
#include <unistd.h>
#include <omp.h>

int main() {
  int x = 123;
  printf("before : x = %d\n", x);
#pragma omp parallel
  {
    int id = omp_get_thread_num();
#pragma omp critical
    x++;
    printf("thread %d : x = %d\n", id, x);
  }
  printf("after : x = %d\n", x);
  return 0;
}


Overwriting omp_critical.c


In [713]:
clang -fopenmp omp_critical.c -o omp_critical
# nvc -mp omp_critical.c -o omp_critical

In [714]:
OMP_NUM_THREADS=30 ./omp_critical

before : x = 123
thread 0 : x = 124
thread 2 : x = 127
thread 12 : x = 130
thread 15 : x = 131
thread 28 : x = 134
thread 19 : x = 137
thread 8 : x = 132
thread 13 : x = 133
thread 10 : x = 145
thread 5 : x = 129
thread 27 : x = 151
thread 18 : x = 139
thread 1 : x = 135
thread 20 : x = 140
thread 3 : x = 125
thread 6 : x = 143
thread 23 : x = 142
thread 14 : x = 144
thread 9 : x = 146
thread 11 : x = 147
thread 24 : x = 148
thread 4 : x = 126
thread 7 : x = 128
thread 29 : x = 136
thread 21 : x = 149
thread 22 : x = 150
thread 25 : x = 152
thread 17 : x = 141
thread 26 : x = 153
thread 16 : x = 138
after : x = 153


# 14. `#pragma omp atomic`
* [#pragma omp atomic](https://www.openmp.org/spec-html/5.0/openmpsu95.html#x126-4840002.17.7) is similar to `#pragma omp critical` but its effect is slightly different and its applicability limited (see below)

# <font color="green"> Problem 11 :  Apply `#pragma omp atomic`</font>
* Add `#pragma omp atomic` to the statement `x++` and observe the result

In [715]:
%%writefile omp_atomic.c
#include <stdio.h>
#include <unistd.h>
#include <omp.h>

int main() {
  int x = 123;
  printf("before : x = %d\n", x);
#pragma omp parallel
  {
    int id = omp_get_thread_num();
#pragma omp atomic
    x++;
    printf("thread %d : x = %d\n", id, x);
  }
  printf("after : x = %d\n", x);
  return 0;
}


Overwriting omp_atomic.c


In [716]:
clang -fopenmp omp_atomic.c -o omp_atomic
# nvc -mp omp_atomic.c -o omp_atomic

In [717]:
OMP_NUM_THREADS=30 ./omp_atomic

before : x = 123
thread 0 : x = 124
thread 15 : x = 130
thread 24 : x = 132
thread 14 : x = 134
thread 28 : x = 136
thread 12 : x = 128
thread 9 : x = 131
thread 2 : x = 142
thread 22 : x = 146
thread 6 : x = 150
thread 26 : x = 153
thread 13 : x = 135
thread 11 : x = 126
thread 8 : x = 125
thread 10 : x = 129
thread 3 : x = 137
thread 4 : x = 138
thread 29 : x = 140
thread 20 : x = 139
thread 1 : x = 143
thread 19 : x = 144
thread 18 : x = 145
thread 25 : x = 141
thread 17 : x = 133
thread 27 : x = 147
thread 5 : x = 151
thread 21 : x = 148
thread 7 : x = 152
thread 23 : x = 149
thread 16 : x = 127
after : x = 153


* The statement that follows this pragma cannot be an arbitrary expression
* See [atomic Construct](https://www.openmp.org/spec-html/5.0/openmpsu95.html#x126-4840002.17.7) for the spec
* Typically, it is an update to a variable, such as
```  
x += expr;
```  
* What is guaranteed by `#pragma omp atomic` is different from what `#pragma omp critical` guarantees
```
#pragma omp atomic
x += expr;
``` 
guarantees that the read and write to _x_ are never intervened by another update labeled `#pragma omp atomic` whereas
```
#pragma omp critical
x += expr;
``` 
guarantees that the entire statement `x += expr` does not overlap with another statement labeled critical.
* When applicable, `#pragma omp atomic` is more efficient than `#pragma omp critical` because the evaluation of expr can overlap

# 15. Reduction clause
* [Reduction](https://www.openmp.org/spec-html/5.0/openmpsu107.html#x140-5800002.19.5) is the best way to resolve race conditions where applicable and it often is
* It is applicable when threads altogether calculate $v = v_0 \oplus v_1 \oplus ... \oplus v_{n-1}$ where $v_i$ can be computed independently and $\oplus$ is an associative operator (such as +)
* In serial loop, this could be written by
```
v = initial value;
for (i = 0; i < n; i++) {
  v_i = ...
  v = v + v_i;
}
```
* If we parallelize the above loop, updating $v$ will result in a race condition
* This can be safely parallelized by introducing `reduction(+ : v)`

# <font color="green"> Problem 12 :  Apply reduction</font>
* Add `reduction` clause to `#pragma omp parallel` below and observe the result

In [718]:
%%writefile omp_reduction.c
#include <stdio.h>
#include <unistd.h>
#include <omp.h>

int main() {
  int x = 123;
  printf("before : x = %d\n", x);
#pragma omp parallel reduction(+ : x)
  {
    int id = omp_get_thread_num();
    x++;
    printf("thread %d : x = %d\n", id, x);
  }
  printf("after : x = %d\n", x);
  return 0;
}


Overwriting omp_reduction.c


In [719]:
clang -fopenmp omp_reduction.c -o omp_reduction
# nvc -mp omp_reduction.c -o omp_reduction

In [720]:
OMP_NUM_THREADS=10 ./omp_reduction

before : x = 123
thread 0 : x = 1
thread 2 : x = 1
thread 8 : x = 1
thread 1 : x = 1
thread 9 : x = 1
thread 3 : x = 1
thread 4 : x = 1
thread 5 : x = 1
thread 6 : x = 1
thread 7 : x = 1
after : x = 133


# 16. How reduction clause works and why it is preferable when applicable
* Where applicable, reduction is generally much faster than using `#pragma omp atomic` or `#pragma omp critical`
* This is because, internally, each thread computes its partial results using a private variable and combines their results only once at the end
* That is, each update, e.g., `x += expr` updates a thread's private version of variable x instead of updating the shared variable
* Omitting details you can think of what reduction clause is doing is to convert something like

```
int x = G;
#pragma omp parallel reduction(+ : x)
{
   ... x += expr; ...
}
  
```

into something like

```
int x = G;
#pragma omp parallel
{
  int x_priv = 0; // (I) initialize a private version of x
  {
   ... x_priv += expr; ...
  }
#pragma omp atomic
  x += x_priv; // (C) combine the partial results in the private variable into the global variable
}
```
* This is valid because of the associativity of the operation

# 17. User-defined reduction
* Reduction is a general concept for efficiently executing many computations of $v_i$'s in parallel, when the final outcome we wish to compute is $v = v_0 \oplus v_1 \oplus ... \oplus v_{n-1}$ 
* It is applicable whenever the order of combining partial results via $\oplus$ does not affect the final outcome (e.g., +)
* Yet the builtin reduction clause of OpenMP can only specify a few builtin operations for a few builtin types (e.g., int, float, etc.)
* You sometimes desire to apply the efficient execution mechanism of the reduction for more general types (perhaps types you defined)
* [User-defined reduction](https://www.openmp.org/spec-html/5.0/openmpsu107.html#x140-5800002.19.5) exists exactly for that
* You need to define an expression to
  * (C) combine two partial results into one (more specifically, combine a partial result assumed to be in a variable omp_in into another variable omp_out)
  * (I) initialize a thread-private version of the variable to which reduction is applied, named omp_priv
* For example, in the case of the builtin + operator, 
  * (C) would be omp_out += omp_in
  * (I) would be omp_priv = 0

## 17-1. Apply a user-defined reduction
* Here is a simple (broken) parallel for loop that is meant to do a reduction on 3-element vector
* Define a reduction with `#pragma omp declare reduction` and apply it to the parallel loop

In [721]:
%%writefile omp_ud_reduction.c
#include <stdio.h>
#include <unistd.h>
#include <math.h>
#include <omp.h>

/* 3-element vector */
typedef struct {
  double a[3];
} vec_t;

/* x += y */
void vec_add(vec_t * x, vec_t * y) {
  for (int i = 0; i < 3; i++) {
    x->a[i] += y->a[i];
  }
}

/* x = {0,0,0} */
void vec_init(vec_t * x) {
  for (int i = 0; i < 3; i++) {
    x->a[i] = 0;
  }
}


/* add an appropriate #pragma omp declare reduction ... here */
#pragma omp declare reduction(vp : vec_t : vec_add(&omp_out, &omp_in)) \
    initializer(vec_init(&omp_priv))
  
int main() {
  vec_t v;
  vec_init(&v);
  double t0 = omp_get_wtime();

  /* add an appropriate reduction clause, so that
     the result is always {10000,10000,10000} */
#pragma omp parallel for reduction(vp : v)
  for (int i = 0; i < 30000; i++) {
    v.a[i % 3]++;
  }
  double t1 = omp_get_wtime();
  printf("ans = {%.1f, %.1f, %.1f} in %f sec\n", v.a[0], v.a[1], v.a[2], t1 - t0);
  return 0;
}


Overwriting omp_ud_reduction.c


In [722]:
clang -fopenmp omp_ud_reduction.c -o omp_ud_reduction
# nvc -mp omp_ud_reduction.c -o omp_ud_reduction

In [723]:
OMP_NUM_THREADS=10 ./omp_ud_reduction

ans = {10000.0, 10000.0, 10000.0} in 0.018463 sec


# <font color="green"> Problem 13 :  Putting them together: calculating an integral</font>
Write an OpenMP program that calculates

$$ \int_0^1 \int_0^1 \sqrt{1 - x^2 - y^2}\,dx\,dy $$

* Use a nested loop to calculate the double integral
* Try work-sharing for, taskloop, recursive tasks to parallelize it
* The result should be close to $\pi/6 = 0.52359..$ (1/8 of the volume of the unit ball)
* Play with the number of infinitesimal intervals for integration and the number of threads so that you can observe a speedup
* As you are using a shared cloud environment, you do not have to be serious about speedup (nearly perfect speedup is unlikely when other students are simultaneously using the same machine and/or the cloud is doing many other stuff (e.g., servicing the page you are looking at right now)

* If you want to work with an editor you are accustomed to rather than web browser, see the `SSH (Remote Login)` section in `pd00_intro.sos` 

In [724]:
%%writefile omp_integral.c
#include <stdio.h>
#include <math.h>
#include <unistd.h>
#include <omp.h>

int main()
{
    // No parallel programming
    long long n0 = 20000;
    double interval0 = 1.0 / n0;
    double result0 = 0.0;
    double t0_beg = omp_get_wtime();
    for (long long i = 0; i < n0; i++) {
        for (long long j = 0; j < n0; j++) {
            if (n0 * n0 - i * i > j * j)
                result0 += interval0 * interval0 * interval0 * \
                            sqrtl(n0 * n0 - i * i - j * j);
        }
    }
    double t0_end = omp_get_wtime();
    printf("regular programming (interval = %f): ans = %.8f in %f sec.\n", interval0, result0, t0_end - t0_beg);

    // Reduction
    long long n1 = 20000;
    double interval1 = 1.0 / n1;
    double result1 = 0.0;
    double t1_beg = omp_get_wtime();
    for (long long i = 0; i < n1; i++) {
#pragma omp parallel for reduction(+ : result1)
        for (long long j = 0; j < n1; j++) {
            if (n1 * n1 - i * i > j * j)
                result1 += interval1 * interval1 * interval1 * \
                            sqrtl(n1 * n1 - i * i - j * j);
        }
    }
    double t1_end = omp_get_wtime();
    printf("reduction (interval = %f): ans = %.8f in %f sec.\n", interval1, result1, t1_end - t1_beg);

    // Reduction
    long long n2 = 100000;
    double interval2 = 1.0 / n2;
    double result2 = 0.0;
    double t2_beg = omp_get_wtime();
    for (long long i = 0; i < n2; i++) {
#pragma omp parallel for reduction(+ : result2)
        for (long long j = 0; j < n2; j++) {
            if (n2 * n2 - i * i > j * j)
                result2 += interval2 * interval2 * interval2 * \
                            sqrtl(n2 * n2 - i * i - j * j);
        }
    }
    double t2_end = omp_get_wtime();
    printf("reduction (interval = %f): ans = %.8f in %f sec.\n", interval2, result2, t2_end - t2_beg);

    return 0;
}

Overwriting omp_integral.c


* Compile it

In [725]:
clang -fopenmp omp_integral.c -o omp_integral -lm
# nvc -mp omp_integral.c -o omp_integral -lm

* and run it

In [726]:
OMP_NUM_THREADS=50 ./omp_integral

regular programming (interval = 0.000050): ans = 0.52363805 in 3.847852 sec.
reduction (interval = 0.000050): ans = 0.52363805 in 0.708320 sec.
reduction (interval = 0.000010): ans = 0.52360663 in 4.360775 sec.


- Because the double integral loop is not a perfectly rectangular for loop (input must **NOT** be negative in square root function), so we can't use parallel `for` or `collapse` in this case. And the execution time of each iteration is very close, and it's simpler to write nested for loop than recursive methods, so I think `task` or `taskloop` is not suitable for this case. So I personally think `reduction` is the most suitable parallem strategy in this case.

- We use [trapezoidal rule](https://en.wikipedia.org/wiki/Trapezoidal_rule) to calculate the approximate integral value, the minimal proportion we set for regular programming is **5*10<sup>-5</sup>**, for parallel programming is **5*10<sup>-5</sup>** and **10<sup>-5</sup>**.

- The real result should be $\pi/6 = 0.52359878..$, the result we got from parallel programming code above is $0.52360663$. With the decrease of the minimal proportion, the result will become more precise and closer to the real value.

- The execution time is clearly showed above, the approximate execution time for regular programming with interval equal to **10<sup>-5</sup>** will probably be 100 seconds in this case, because the time complexity for this double integral algorithm is **O(n<sup>2</sup>)**, the execution time has to multiply 5 * 5 = 25.